# Load Data

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maxan\OneDrive\Desktop\0. Personal Projects\market-intelligence-pipeline")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    calendar_df["date"] = pd.to_datetime(calendar_df["date"])

    calendar_df = calendar_df.sort_values(["ticker", "date"])

    print(f"[DONE] Prepared and sorted table")

    return calendar_df

In [5]:
df = pull_calendar_table(
    table_name = "price_features", 
    calendar = CALENDAR
    )

print(f"[STEP] Loaded price_features table selecting {CALENDAR} columns")
print(df.head(10))
print(df.columns)

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[DONE] Prepared and sorted table
[STEP] Loaded price_features table selecting no_calendar columns
      ticker       date        open        high         low       close  \
0        GLD 2018-01-02  124.660004  125.180000  124.389999  125.150002   
1        GLD 2018-01-03  125.050003  125.089996  124.099998  124.820000   
8585     GLD 2018-01-04  124.889999  125.849998  124.739998  125.459999   
4299     GLD 2018-01-05  124.930000  125.480003  124.830002  125.330002   
2        GLD 2018-01-08  125.199997  125.320000  124.900002  125.309998   
7160     GLD 2018-01-09  124.489998  124.860001  124.230003  124.730003   
4300     GLD 2018-01-10  125.169998  125.309998  124.720001  125.029999   
10109    GLD 2018-01-11  125.370003  125.660004  125.250000  125.440002   
10110    GLD 2018-01-12  126.010002  127.129997  125.80

# Target Next Day Return

In [31]:
df_1 = df.copy()

target_col = f"target_next_day_return_{CALENDAR}"

tndr_series = df_1[target_col]

tndr_summary = tndr_series.agg([
    "mean",
    "median",
    "std",
    "min",
    "max",
    "skew",
    "kurt"
])

tndr_summary.loc["p1"] = tndr_series.quantile(0.01)
tndr_summary.loc["p99"] = tndr_series.quantile(0.99)

# .T means transpose
tndr_summary_df = tndr_summary.to_frame().T
tndr_summary_df.index = ["Overall"]

print("Overall TNDR Summary Across All Assets:")
display(tndr_summary_df)



ticker_summary_rows = []

tndr_ticker_df = df_1[["ticker",target_col]]

for ticker, group in tndr_ticker_df.groupby("ticker"):
    tndr_ticker_summary = group[target_col].agg([
        "mean",
        "median",
        "std",
        "min",
        "max",
        "skew",
        "kurt"
    ])

    tndr_ticker_summary.loc["p1"] = group[target_col].quantile(0.01)
    tndr_ticker_summary.loc["p99"] = group[target_col].quantile(0.99)

    tndr_ticker_summary = tndr_ticker_summary.to_frame().T
    tndr_ticker_summary.index = [ticker]

    ticker_summary_rows.append(tndr_ticker_summary)

tndr_ticker_summary_df = pd.concat(ticker_summary_rows)

print("Ticker TNDR Summary:")
display(tndr_ticker_summary_df)



tndr_date_df = df_1[["date", target_col]]
tndr_date_df["year"] = df["date"].dt.year

tndr_summary_by_date = (
    tndr_date_df
    .groupby("year")[target_col]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
        skew="skew",
        kurt="kurt",
        p1=lambda x: x.quantile(0.01),
        p99=lambda x: x.quantile(0.99)
    )
)

# Removes the blank row.
tndr_summary_by_date.index.name = None

print("Year TNDR Summary:")
display(tndr_summary_by_date)



Overall TNDR Summary Across All Assets:


,mean,median,std,min,max,skew,kurt,p1,p99
Overall,0.001152,0.00057,0.024132,-0.213016,0.47091,1.720056,32.236225,-0.062637,0.078408


Ticker TNDR Summary:


,mean,median,std,min,max,skew,kurt,p1,p99
GLD,0.000564,0.000619,0.010588,-0.102742,0.063587,-0.636367,7.305299,-0.030544,0.027924
MU,0.002091,0.001012,0.033101,-0.198186,0.192916,0.175578,3.593143,-0.079828,0.097890
NKE,0.000087,0.000000,0.021387,-0.199809,0.155314,-0.268748,12.821108,-0.057467,0.055998
RPI.L,0.002499,-0.001033,0.047394,-0.139640,0.470910,3.267675,26.666113,-0.095784,0.149991
SNDK,0.014118,0.008176,0.064327,-0.213016,0.286494,0.345475,2.506943,-0.155648,0.174566
SPY,0.000614,0.000900,0.012091,-0.109424,0.105019,-0.294439,13.116533,-0.033655,0.028521
TLT,-0.000013,0.000123,0.009738,-0.066683,0.075196,0.161246,5.043367,-0.023337,0.024603


Year TNDR Summary:


,mean,median,std,min,max,skew,kurt,p1,p99
2018,-0.000017,0.000172,0.017193,-0.098708,0.113706,0.001590,8.024218,-0.057854,0.050369
2019,0.001260,0.001203,0.014912,-0.110905,0.133415,0.468563,11.081058,-0.042178,0.047834
2020,0.001189,0.001438,0.023514,-0.198186,0.151751,-0.290356,10.326229,-0.070029,0.073898
2021,0.000550,0.000440,0.015220,-0.064417,0.155314,0.878589,12.715144,-0.046496,0.044446
2022,-0.000966,-0.001154,0.020399,-0.128081,0.121790,0.043886,3.784291,-0.058151,0.054146
2023,0.000705,0.000270,0.014758,-0.118257,0.086288,0.403832,7.196704,-0.035409,0.047948
2024,0.000680,0.000906,0.020578,-0.199809,0.157480,-0.274193,17.178803,-0.060185,0.060702
2025,0.002293,0.000910,0.031859,-0.213016,0.286494,0.653459,12.552230,-0.080006,0.114199
2026,0.005784,0.000695,0.047252,-0.159538,0.470910,2.405737,18.375886,-0.107236,0.156095


Central tendency - mean vs median:
- Overall: 
    - General positive next day returns since both are positive.
- Ticker: 
    - RPI.L stands out with positive mean but negative median indicaiting average may be driven by a few large winners. 
    - TLT also stands out with negative mean but positive median indicating mostly positive returns bu occasianl large losses. 
    - Some assets show more positive mean compared to median indicating returns driven by few large returns.
- Year: 
    - 2022 indicates mostly negative returns while 2025 shows mostly postiive returns but occasional large spikes - possibly in one or a few assets.

Return to risk - mean vs volatility:
- Overall:
    - Across this period we have seen poor returns given the level of risk with a much higher volatility compared to returns.
- Ticker:
    - TLT shows negative returns wiht respect to its risk although low.
    - SNDK and RPI.L have shown very high volatility. MU shows a better balance.
- Year:
    - 2026 was a good year with high returns although high risk.
    - 2019 was the only year to show relatively good balance between risk and return.

# Target Direction

# Class Balance for Target Direction

# Percent of Up vs Down Days

# Target Behaviour By Ticker

# Target Noise

# Modelling Implication